[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 00](README.md)

# Entorno reproducible y diagnóstico

**Tema:** 00 · **Sesiones:** 1, 2 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo demostrar que una práctica puede construirse y repetirse en el equipo disponible?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Antes de paralelizar, hay que saber qué equipo y qué herramientas ejecutarán el programa. Este notebook enseña a convertir el ambiente en evidencia verificable.

**Prerrequisitos.**

- Manejo básico de terminal y archivos.
- Diferencia elemental entre código fuente y programa ejecutable.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Distinguir plataforma, toolchain, runtime y dependencia.
- Registrar evidencia mínima del sistema sin confundir disponibilidad con compatibilidad.
- Ejecutar el preflight y leer sus informes antes de iniciar una práctica.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Un entorno reproducible declara versiones, arquitectura y comandos; no se reduce a una lista de paquetes.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

El manifiesto de plataforma describe lo observado. La política del ejercicio determina si ese equipo es compatible.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

CMake configura y CTest verifica corrección; las mediciones de rendimiento se realizan solo después de superar las pruebas.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- plataforma — sistema, arquitectura y hardware observados
- toolchain — compilador, enlazador, bibliotecas y herramientas
- runtime — soporte que participa durante la ejecución


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Ruta Reproducible

![Ruta reproducible desde la hipótesis hasta el informe](../../images/ruta-reproducible.svg)

**Cómo leerlo.** Sigue las flechas de izquierda a derecha. La validación aparece antes de la medición porque un resultado rápido pero incorrecto no constituye evidencia de rendimiento.

### Capas Toolchain

![Capas de código, compilador, runtime, sistema y hardware](../../images/capas-toolchain.svg)

**Cómo leerlo.** Lee de arriba hacia abajo: cada capa añade condiciones que una macro o una bandera aislada no puede demostrar por sí sola.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "00"
NOTEBOOK = "00_entorno/00_entorno_reproducible.ipynb"
assert (ROOT / "curso" / "notebooks" / "00_entorno" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Inventario local

**Situación.** Se inspeccionan datos portables del intérprete y la presencia de herramientas sin instalar ni modificar el sistema.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
import platform, shutil, sys
inventory = {
    "python": sys.version.split()[0],
    "os": platform.system(),
    "release": platform.release(),
    "machine": platform.machine(),
    "logical_cpus": __import__("os").cpu_count(),
    "cmake": shutil.which("cmake"),
    "ctest": shutil.which("ctest"),
    "cc": shutil.which("cc"),
    "cxx": shutil.which("c++"),
}
assert inventory["logical_cpus"] and inventory["logical_cpus"] > 0
for key, value in inventory.items():
    print(f"{key:12}: {value}")


### Explicación del resultado

Una ruta ausente se reporta como evidencia diagnóstica; no se sustituye por una afirmación de soporte.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Configuración declarada

**Situación.** Se extraen las versiones canónicas del toolchain y se comprueba que las claves esenciales estén presentes.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
import re
toolchain = (ROOT / "config" / "course-toolchain.cmake").read_text(encoding="utf-8")
pairs = dict(re.findall(r'set\((COURSE_[A-Z0-9_]+) "([^"]+)"', toolchain))
required = {"COURSE_GCC_VERSION", "COURSE_CXX_STANDARD", "COURSE_MPI_VERSION", "COURSE_CUDA_VERSION", "COURSE_PYTHON_VERSION"}
assert required <= pairs.keys(), required - pairs.keys()
for key in sorted(required):
    print(f"{key}={pairs[key]}")


### Lectura razonada

La versión declarada es un requisito; el manifiesto del equipo permite contrastarla con la versión observada.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué puede concluirse si CMake existe, pero el compilador requerido o el runtime no coincide con la política?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Ejecutar `python3 validation/preflight.py` desde la raíz.
2. Conservar los JSON de `build/validation/preflight/`.
3. Explicar qué comprobó cada etapa y qué no demuestra todavía.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Continuar aunque el preflight falle.
- Confundir arquitectura con fabricante de CPU.
- Afirmar soporte de GPU porque `nvcc` está instalado sin ejecutar en un dispositivo.


## Criterios de aceptación

- Preflight con código de salida cero.
- Manifiesto de plataforma adjunto al informe.
- Limitaciones del equipo descritas explícitamente.


## Síntesis

- La pregunta que debes poder responder es: **¿Cómo demostrar que una práctica puede construirse y repetirse en el equipo disponible?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Protocolo de reproducibilidad](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md)
- [Configuración del curso](../../../config/README.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 00](README.md)
